# Keras/TensorFlow — Chapter 9: Project — Multiclass Classification of Iris Species


## 1. Bài toán và dữ liệu

- Bộ dữ liệu **Iris**, mục tiêu accuracy tham chiếu **95–97%**. Nạp bằng `pandas` (nhãn là chuỗi).

## 2. Mã hoá nhãn: LabelEncoder + to_categorical

- Đi qua **2 bước**: `LabelEncoder` (chuỗi → số nguyên 0/1/2) rồi `to_categorical()` (số nguyên → one-hot).

## 3. Model + kỹ thuật mới: bọc Keras model bằng scikit-learn (`KerasClassifier`)

- Kiến trúc tối giản: 4 input → 8 hidden (ReLU) → 3 output.
- Model phải viết dưới dạng **hàm trả về model đã compile** (`baseline_model()`).
- `KerasClassifier(model=baseline_model, epochs=200, batch_size=5)` (từ thư viện **SciKeras**) — bọc hàm đó thành một **estimator kiểu scikit-learn**, cho phép dùng thẳng các công cụ đánh giá của scikit-learn (`cross_val_score`) lên model Keras mà không cần tự viết vòng lặp k-fold thủ công như Chapter 8.

## 4. Đánh giá: cross_val_score thay vì tự viết vòng lặp

- `kfold = KFold(n_splits=10, shuffle=True)` rồi `cross_val_score(estimator, X, dummy_y, cv=kfold)` — 1 dòng thay thế toàn bộ vòng lặp `for train, test in kfold.split(...)`.
- **Kết quả**: **97.33% ± 4.42%** — đạt đúng mục tiêu 95-97%, độ lệch chuẩn nhỏ vì bài toán Iris dễ và dữ liệu sạch hơn Pima Diabetes.


## 5. Vận dụng


**9.1–9.8** — load Iris + one-hot encode + định nghĩa model qua hàm + đánh giá bằng KerasClassifier + 10-fold cross-validation

In [3]:
# multi-class classification with Keras
import pandas
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
# load dataset
dataframe = pandas.read_csv("iris.csv", header=None)
dataset = dataframe.values
X = dataset[:,0:4].astype(float)
Y = dataset[:,4]
# encode class values as integers
encoder = LabelEncoder()
encoder.fit(Y)
encoded_Y = encoder.transform(Y)
# convert integers to dummy variables (i.e. one-hot encoded)
dummy_y = to_categorical(encoded_Y)

# define baseline model
def baseline_model():
    # create model
    model = Sequential()
    model.add(Dense(8, input_shape=(4,), activation='relu'))
    model.add(Dense(3, activation='softmax'))
    # Compile model
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

estimator = KerasClassifier(model=baseline_model, epochs=200, batch_size=5, verbose=0)
kfold = KFold(n_splits=10, shuffle=True)
results = cross_val_score(estimator, X, dummy_y, cv=kfold)
print("Baseline: %.2f%% (%.2f%%)" % (results.mean()*100, results.std()*100))


Baseline: 96.67% (5.37%)
